In [1]:
import pandas as pd, xarray as xr, numpy as np, matplotlib as mpl, geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt 
import pickle

from functions import calc_z_scores, calc_N_and_f_fast, aggregate_by_country, \
                                  build_country_weights, verify_population_coverage

In [4]:
## File path shortcuts
basepath = '/home/msaenger/Perceptions_2025/cleaned_code/'
cpath = basepath + 'CIDER_output/'

n_members = 500

newpath = basepath + f'{n_members}_members/'
print(newpath)

/home/msaenger/Perceptions_2025/cleaned_code/500_members/


In [5]:
ssp_plus_invert = xr.open_dataset(newpath + 
                    f'SSP370_plus_InVERT_{n_members}member_ensemble_TREFHT_2015_to_2065.nc')

In [6]:
ssp_plus_invert

<xarray.Dataset> Size: 44GB
Dimensions:   (time: 600, lat: 96, ensemble: 500, lon: 192)
Coordinates:
  * time      (time) object 5kB 2015-01-16 00:00:00 ... 2064-12-16 00:00:00
  * lat       (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
    month     (time) int64 5kB ...
  * ensemble  (ensemble) int64 4kB 0 1 2 3 4 5 6 ... 493 494 495 496 497 498 499
  * lon       (lon) float64 2kB -178.1 -176.2 -174.4 ... 176.2 178.1 180.0
Data variables:
    TREFHT    (time, lat, lon, ensemble) float64 44GB ...
    gmean     (time, ensemble) float64 2MB ...

### Calculate z-scores at each grid cell 

In [7]:
## Define baseline time period
baseline_start = '2015-01'
baseline_end = '2024-12'

## Select baseline period from combined dataset
baseline = ssp_plus_invert.sel(time=slice(baseline_start, baseline_end))

In [8]:
## 2020 population data, already on the 96x192 model grid
pop = xr.open_dataarray(basepath + 'gpw-v4-population-count-rev11_totpop_96x192.nc').sel(time=2020)

countries_zip_path = basepath + 'natural_earth_data/ne_50m_admin_0_countries.zip'

country_weights_dict = build_country_weights(pop, countries_zip_path,
                keep_as_sovereign=['Palestine'])

## Sanity check that every grid cell's population is accounted for in the country rows
verify_population_coverage(country_weights_dict)

Folding 41 non-sovereign territories into their sovereign state, e.g.: ['Aland -> Finland', 'American Samoa -> United States of America', 'Anguilla -> United Kingdom', 'Aruba -> Netherlands', 'Ashmore and Cartier Islands -> Australia', 'Bermuda -> United Kingdom', 'British Indian Ocean Territory -> United Kingdom', 'British Virgin Islands -> United Kingdom', 'Cayman Islands -> United Kingdom', 'Cook Islands -> New Zealand'] ...
Reassigned 12365 grid points with no direct polygon match to their nearest sovereign country (6.752e+08 people, 8.42% of global population).
Dropping 7 countries with zero population weight: ['Antarctica', 'El Salvador', 'Barbados', 'Saint Vincent and the Grenadines', 'Saint Lucia', 'Israel', 'Malta']
Smallest remaining population weight: 3.291e+00
Population coverage: 100.000000% of total global population weight is accounted for across all kept country rows.
Dropping 3 EU countries with no grid coverage even after nearest-country fallback: ['Belgium', 'Luxembo

{'population_total': 8016736089.992069,
 'population_accounted': 8016736089.992069,
 'coverage_frac': 1.0,
 'passed': True}

In [9]:
W_norm_eu  = country_weights_dict['W_norm_eu']   
countries_idx_eu = country_weights_dict['countries_idx_eu']
country_da = country_weights_dict['country_da']
weights_2d = country_weights_dict['weights_2d']       
country_flat = country_weights_dict['country_flat']    
weights_flat_vals = country_weights_dict['weights_flat_vals']
folded_non_sovereign = country_weights_dict['folded_non_sovereign']
dropped_zero_pop = country_weights_dict['dropped_zero_pop']
population_total = country_weights_dict['population_total']
population_accounted = country_weights_dict['population_accounted']
population_coverage_frac = country_weights_dict['population_coverage_frac']

In [10]:
print('Computing z scores')

z_scores = calc_z_scores(ds=ssp_plus_invert, var_name='TREFHT', 
                         baseline=baseline)

Computing z scores


In [11]:
def calc_z_scores_vectorized(ds, var_name, baseline, monthly_std=True):
    """Vectorized replacement."""
    baseline_monthly_mean = baseline[var_name].mean('ensemble').groupby('time.month').mean('time')
    anomaly = ds[var_name].groupby('time.month') - baseline_monthly_mean

    if monthly_std:
        baseline_monthly_std = baseline[var_name].std('ensemble').groupby('time.month').mean('time')
        z_scores_da = anomaly.groupby('time.month') / baseline_monthly_std
    else:
        baseline_std = baseline[var_name].std('ensemble').mean('time')
        z_scores_da = anomaly / baseline_std

    return z_scores_da

z_scores_test = calc_z_scores_vectorized(ssp_plus_invert, 'TREFHT', baseline)


In [12]:
z_scores.equals(z_scores_test)

True

### Compute f at each grid cell and aggregate by country

In [ ]:
tau = 12
delta_t = 1

k_vals = [625, 650, 675, 700, 725, 750, 775, 800, 825, 850, 875, 900, 925, 950, 975, 1000]

#[50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 
          #300, 325, 350, 375, 400, 425, 450, 475, 500, 
          #525, 550, 575, 600]

f_dict = {}

print('Computing f')

for k in k_vals:
    print(k)
    
    N, f = calc_N_and_f_fast(z_scores, k, tau, delta_t)
    
    f_dict[k] = f

In [ ]:
print('Aggregating f by country')

f_country_dict = {}
for k in k_vals:
    print(k)
    f_country_dict[k] = aggregate_by_country(f_dict[k], W_norm_eu, countries_idx_eu)

In [ ]:
print('Saving dict')

## Save dictionary of country-aggregated f vals for multiple k

with open(newpath + 'f_by_country_no_SAI_625_to_1000.pickle', 'wb') as handle:
    pickle.dump(f_country_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
print('Done')